## 1. Import Necessary Libraries

In [424]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from datetime import datetime

## 2. Import Dataset

In [425]:
creditCard_data = pd.read_csv(filepath_or_buffer="data/credit_card_clean.csv")

## 3. Data Understanding

### Exploring the variable
* ID : ID of each client

* LIMIT_BAL: Amount of given credit in NT dollars (includes individual and family/supplementary credit)

* SEX: Gender (1=male, 2=female)

* EDUCATION: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)

* MARRIAGE: Marital status (1=married, 2=single, 3=others)

* AGE: Age in years

* PAY_1: Repayment status in September, 2005 (-2 = No consumption, -1 = paid in full, 0 = use of revolving credit (paid minimum only), 1 = payment delay for one month, 2 = payment delay for two months, ... 8 = payment delay for eight months, 9 = payment delay for nine months and above)

* PAY_2: Repayment status in August, 2005 (scale same as above)

* PAY_3: Repayment status in July, 2005 (scale same as above)

* PAY_4: Repayment status in June, 2005 (scale same as above)

* PAY_5: Repayment status in May, 2005 (scale same as above)

* PAY_6: Repayment status in April, 2005 (scale same as above)

* BILL_AMT1: Amount of bill statement in September, 2005 (NT dollar)

* BILL_AMT2: Amount of bill statement in August, 2005 (NT dollar)

* BILL_AMT3: Amount of bill statement in July, 2005 (NT dollar)

* BILL_AMT4: Amount of bill statement in June, 2005 (NT dollar)

* BILL_AMT5: Amount of bill statement in May, 2005 (NT dollar)

* BILL_AMT6: Amount of bill statement in April, 2005 (NT dollar)

* PAY_AMT1: Amount of previous payment in September, 2005 (NT dollar)

* PAY_AMT2: Amount of previous payment in August, 2005 (NT dollar)

* PAY_AMT3: Amount of previous payment in July, 2005 (NT dollar)

* PAY_AMT4: Amount of previous payment in June, 2005 (NT dollar)

* PAY_AMT5: Amount of previous payment in May, 2005 (NT dollar)

* PAY_AMT6: Amount of previous payment in April, 2005 (NT dollar)

#### Target variable
default.payment.next.month: Default payment (1=yes, 0=no)

###### Note: NT is Taiwain Dollars

In [426]:
creditCard_data.isna().sum()

ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
DEFAULT      0
dtype: int64

In [427]:
creditCard_data.describe()

,ID,LIMIT_BAL,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,-0.291100,51223.330900,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,9.217904,1.123802,1.197186,1.196868,1.169139,1.133187,1.149988,73635.860576,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-165580.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,3558.750000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22381.500000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,67091.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,79.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,964511.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


In [428]:
creditCard_data.describe(include="all")

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
count,30000.000000,30000.000000,30000,30000,30000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
unique,NaN,NaN,2,4,3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,female,university,single,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,18112,14030,15964,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,15000.500000,167484.322667,NaN,NaN,NaN,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,NaN,NaN,NaN,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,NaN,NaN,NaN,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,NaN,NaN,NaN,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,NaN,NaN,NaN,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,NaN,NaN,NaN,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000


In [429]:
creditCard_data.dtypes

ID             int64
LIMIT_BAL    float64
SEX              str
EDUCATION        str
MARRIAGE         str
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

In [430]:
creditCard_data.shape

(30000, 25)

## 4. Data Preperation

In [431]:
creditCard_data.columns.tolist()

['ID',
 'LIMIT_BAL',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_1',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'DEFAULT']

In [432]:
creditCard_data = creditCard_data.drop(labels=["ID"], axis=1)

In [433]:
creditCard_data.sample(10)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
7751,90000.0,female,university,other,40,1,2,0,0,0,...,61778.0,68248.0,66997.0,3000.0,2773.0,2402.0,7660.0,0.0,2711.0,0
11457,130000.0,male,other,single,46,2,0,0,0,0,...,37834.0,37429.0,36991.0,2475.0,1600.0,1300.0,1400.0,1400.0,2000.0,0
22167,210000.0,female,university,single,27,0,0,0,0,0,...,27071.0,27817.0,28401.0,2000.0,1800.0,2000.0,1500.0,1031.0,1191.0,0
2093,80000.0,female,other,single,34,1,2,-1,-1,-1,...,1261.0,2522.0,1261.0,0.0,1261.0,1261.0,2522.0,0.0,780.0,0
3271,160000.0,male,grad,married,40,0,0,0,0,0,...,36100.0,21707.0,7928.0,5002.0,5000.0,14400.0,10000.0,4000.0,0.0,0
12514,90000.0,female,university,single,29,0,0,0,0,0,...,46544.0,48266.0,49926.0,4073.0,5021.0,3000.0,3000.0,3000.0,2000.0,0
9095,200000.0,male,university,married,36,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
15592,30000.0,male,university,married,53,0,0,0,0,0,...,28884.0,29523.0,30672.0,1427.0,2973.0,1500.0,1100.0,2000.0,1000.0,0
19575,210000.0,female,grad,married,31,0,0,0,0,0,...,119579.0,120318.0,107122.0,7505.0,7072.0,4009.0,8000.0,10000.0,12000.0,0
3646,220000.0,male,highschool,married,41,-2,-2,-2,-2,-2,...,7229.0,12806.0,35433.0,34415.0,14565.0,7229.0,12806.0,35433.0,11405.0,0


In [434]:
le =  LabelEncoder()
creditCard_data["SEX"] = le.fit_transform(creditCard_data["SEX"])
creditCard_data["EDUCATION"] = le.fit_transform(creditCard_data["EDUCATION"])
creditCard_data["MARRIAGE"] = le.fit_transform(creditCard_data["MARRIAGE"])

In [435]:
creditCard_data.sample(5)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
22979,150000.0,0,3,0,40,-1,-1,-1,-1,-1,...,3307.0,2894.0,0.0,0.0,7937.0,3307.0,2894.0,0.0,0.0,0
21060,150000.0,1,1,2,29,-1,-1,2,-1,-1,...,1300.0,936.0,901.0,3431.0,0.0,1300.0,936.0,901.0,911.0,0
1916,110000.0,0,3,2,28,0,0,0,-2,-2,...,0.0,0.0,0.0,2000.0,0.0,0.0,0.0,0.0,0.0,0
16161,140000.0,0,3,2,32,0,0,0,0,0,...,137629.0,120147.0,122247.0,5004.0,5000.0,5000.0,4300.0,5000.0,5158.0,0
6979,340000.0,0,1,2,57,0,0,0,0,0,...,104477.0,54351.0,25346.0,1880.0,3819.0,2073.0,5264.0,2571.0,53860.0,0


In [436]:
creditCard_data.dtypes

LIMIT_BAL    float64
SEX            int64
EDUCATION      int64
MARRIAGE       int64
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

## 5. Model Building

In [437]:
X = creditCard_data.drop(labels=["DEFAULT"], axis=1)
X.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,20000.0,0,3,0,24,2,2,-1,-1,-2,...,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0
1,120000.0,0,3,2,26,-1,2,0,0,0,...,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0
2,90000.0,0,3,2,34,0,0,0,0,0,...,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0
3,50000.0,0,3,0,37,0,0,0,0,0,...,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0
4,50000.0,1,3,0,57,-1,0,-1,0,0,...,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0


In [438]:
y = creditCard_data.DEFAULT
y.head()

0    1
1    1
2    0
3    0
4    0
Name: DEFAULT, dtype: int64

In [439]:
print(f"X_shape: {X.shape}")
print(f"y_shape: {y.shape}")

X_shape: (30000, 23)
y_shape: (30000,)


In [440]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=123, shuffle=True)

In [441]:
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (24000, 23)
X_test: (6000, 23)
y_train: (24000,)
y_test: (6000,)


## 6. Model Training

#### Training Time Analyss

In [442]:
adb_classifier = AdaBoostClassifier(n_estimators=100)
gb_classifier = GradientBoostingClassifier(n_estimators=100)
XGBM_classifier = XGBClassifier(n_estimators=100)
LGBM_classifier = LGBMClassifier(n_estimators=100)

In [443]:
%%time
adb_classifier.fit(X_train, y_train)
print()


CPU times: total: 2.05 s
Wall time: 2.06 s


In [444]:
%%time
gb_classifier.fit(X_train, y_train)
print()


CPU times: total: 4.64 s
Wall time: 4.65 s


In [445]:
%%time
XGBM_classifier.fit(X_train, y_train)
print()


CPU times: total: 1.58 s
Wall time: 88.3 ms


In [446]:
%%time
LGBM_classifier.fit(X_train, y_train)
print()


CPU times: total: 2.03 s
Wall time: 98.9 ms


## 7. Model Testing

In [447]:
# y_pred_train_adb = adb_classifier.predict(X_train)
# y_pred_train_gb = gb_classifier.predict(X_train)
# y_pred_train_xgbm = XGBM_classifier.predict(X_train)
# y_pred_train_lgbm = LGBM_classifier.predict(X_train)

In [448]:
y_pred_test_adb = adb_classifier.predict(X_test)
y_pred_test_gb = gb_classifier.predict(X_test)
y_pred_test_xgbm = XGBM_classifier.predict(X_test)
y_pred_test_lgbm = LGBM_classifier.predict(X_test)

## 8. Model Evaluation

In [449]:
print(f"Accuracy Score:")
print(f"--------------------")
print(f"ADA_Boost: {accuracy_score(y_test, y_pred_test_adb)}")
print(f"GB: {accuracy_score(y_test, y_pred_test_gb)}")
print(f"XGBM: {accuracy_score(y_test, y_pred_test_xgbm)}")
print(f"LGBM: {accuracy_score(y_test, y_pred_test_lgbm)}")
print(f"==============================================================")
print(f"")
print(f"Precision Score:")
print(f"--------------------")
print(f"ADA_Boost: {precision_score(y_test, y_pred_test_adb)}")
print(f"GB: {precision_score(y_test, y_pred_test_gb)}")
print(f"XGBM: {precision_score(y_test, y_pred_test_xgbm)}")
print(f"LGBM: {precision_score(y_test, y_pred_test_lgbm)}")
print(f"==============================================================")
print(f"")
print(f"Recall Score:")
print(f"--------------------")
print(f"ADA_Boost: {recall_score(y_test, y_pred_test_adb)}")
print(f"GB: {recall_score(y_test, y_pred_test_gb)}")
print(f"XGBM: {recall_score(y_test, y_pred_test_xgbm)}")
print(f"LGBM: {recall_score(y_test, y_pred_test_lgbm)}")
print(f"==============================================================")
print(f"")
print(f"Confustion Matrix:")
print(f"--------------------")
print(f"ADA_Boost: \n{confusion_matrix(y_test, y_pred_test_adb)}")
print(f"----------------------------")
print(f"GB: \n{confusion_matrix(y_test, y_pred_test_gb)}")
print(f"----------------------------")
print(f"XGBM: \n{confusion_matrix(y_test, y_pred_test_xgbm)}")
print(f"----------------------------")
print(f"LGBM: \n{confusion_matrix(y_test, y_pred_test_lgbm)}")
print(f"----------------------------")

Accuracy Score:
--------------------
ADA_Boost: 0.8208333333333333
GB: 0.823
XGBM: 0.8158333333333333
LGBM: 0.825

Precision Score:
--------------------
ADA_Boost: 0.7
GB: 0.6911976911976911
XGBM: 0.6520547945205479
LGBM: 0.698140200286123

Recall Score:
--------------------
ADA_Boost: 0.33232856066315
GB: 0.36096458176337604
XGBM: 0.35870384325546345
LGBM: 0.3677467972871138

Confustion Matrix:
--------------------
ADA_Boost: 
[[4484  189]
 [ 886  441]]
----------------------------
GB: 
[[4459  214]
 [ 848  479]]
----------------------------
XGBM: 
[[4419  254]
 [ 851  476]]
----------------------------
LGBM: 
[[4462  211]
 [ 839  488]]
----------------------------


In [450]:
print(f"ADA Boost:")
print(f"-------------")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_test_adb):.3%}")
print(f"Precision Score: {precision_score(y_test, y_pred_test_adb):.3%}")
print(f"Recall Score: {recall_score(y_test, y_pred_test_adb):.3%}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_test_adb)}")
print(f"==============================================================")
print(f"")
print(f"GB Boost:")
print(f"-------------")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_test_gb):.3%}")
print(f"Precision Score: {precision_score(y_test, y_pred_test_gb):.3%}")
print(f"Recall Score: {recall_score(y_test, y_pred_test_gb):.3%}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_test_gb)}")
print(f"==============================================================")
print(f"")
print(f"XGBM Boost:")
print(f"-------------")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_test_xgbm):.3%}")
print(f"Precision Score: {precision_score(y_test, y_pred_test_xgbm):.3%}")
print(f"Recall Score: {recall_score(y_test, y_pred_test_xgbm):.3%}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_test_xgbm)}")
print(f"==============================================================")
print(f"")
print(f"LGBM Boost:")
print(f"-------------")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_test_lgbm):.3%}")
print(f"Precision Score: {precision_score(y_test, y_pred_test_lgbm):.3%}")
print(f"Recall Score: {recall_score(y_test, y_pred_test_lgbm):.3%}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_test_lgbm)}")

ADA Boost:
-------------
Accuracy Score: 82.083%
Precision Score: 70.000%
Recall Score: 33.233%
Confusion Matrix:
[[4484  189]
 [ 886  441]]

GB Boost:
-------------
Accuracy Score: 82.300%
Precision Score: 69.120%
Recall Score: 36.096%
Confusion Matrix:
[[4459  214]
 [ 848  479]]

XGBM Boost:
-------------
Accuracy Score: 81.583%
Precision Score: 65.205%
Recall Score: 35.870%
Confusion Matrix:
[[4419  254]
 [ 851  476]]

LGBM Boost:
-------------
Accuracy Score: 82.500%
Precision Score: 69.814%
Recall Score: 36.775%
Confusion Matrix:
[[4462  211]
 [ 839  488]]
